In [0]:
# 1) compute embeddings and write to a table with 'embedding' column
from pyspark.sql import SparkSession
import pandas as pd
from databricks_langchain import DatabricksEmbeddings

spark = SparkSession.builder.getOrCreate()
embedder = DatabricksEmbeddings(endpoint="/serving-endpoints/databricks-gte-large-en/invocations")  # or your embedding endpoint

# read your KB docs table
kb_table = "demo.retail_media_kb_docs"   # must exist and have id, content
pdf = spark.table(kb_table).toPandas()

# compute embeddings (batch if large)
pdf["embedding"] = pdf["content"].apply(lambda t: embedder.embed_query(t))

# write back to a new table with embedding column
out_table = "demo.retail_media_kb_docs_with_embeddings"
spark.createDataFrame(pdf).write.mode("overwrite").saveAsTable(out_table)
print("Wrote table with embeddings:", out_table)


In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
%pip install langchain-core databricks-langchain langgraph-supervisor mlflow plotly

In [0]:
# langgraph_with_demo_retail_media_kb.py
"""
Databricks-native LangGraph supervisor using demo.retail_media as KB source.
- Builds a short KB docs table demo.retail_media_kb_docs by concatenating columns
- (Optionally) creates a Delta Sync vector index with managed embeddings
- Runs the supervisor with KB_CONTEXT prepended (no Genie)
Run inside Databricks.
"""

import json
import time
from typing import List, Dict, Any
from uuid import uuid4

from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws, col
import pandas as pd

# Databricks SDK
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain import ChatDatabricks, DatabricksFunctionClient, UCFunctionToolkit, set_uc_function_client

# LangGraph / agent imports
from langchain_core.runnables import Runnable
from langchain.agents import create_agent
from langgraph.graph.state import CompiledStateGraph
from langgraph_supervisor import create_supervisor

# Pydantic
from pydantic import BaseModel

# ---------------- CONFIG (edit if needed) ----------------
SOURCE_TABLE = "demo.retail_media"                         # your real data table
KB_DOCS_TABLE = "demo.retail_media_kb_docs"                # will be created/overwritten
INDEX_NAME = "demo.retail_media_kb_index"                  # vector index name
LLM_ENDPOINT_NAME = "databricks-claude-3-7-sonnet"         # replace if different
KB_TOP_K = 3                                               # top-k retrieval
EMBED_MODEL = "databricks-gte-large"                       # managed embedding model
# ---------------------------------------------------------

# Spark + clients
spark = SparkSession.builder.getOrCreate()
vclient = VectorSearchClient()
uc_client = DatabricksFunctionClient()
set_uc_function_client(uc_client)

# ---------------- small models for agent registration ----------------
class ServedSubAgent(BaseModel):
    endpoint_name: str
    name: str
    task: str
    description: str

class InCodeSubAgent(BaseModel):
    tools: List[str]
    name: str
    description: str

# ---------------- helper: build KB docs table from source ----------------
def build_kb_docs_from_source(src_table: str = SOURCE_TABLE, dest_table: str = KB_DOCS_TABLE, max_rows: int | None = 5000):
    """
    Reads the source table, concatenates a set of columns into a single 'content' text
    and writes a small KB docs table (id, content). Overwrites dest_table.
    Keeps up to max_rows rows (for initial indexing — adjust as needed).
    """
    # read source; limit columns to a reasonable length
    df = spark.table(src_table)
    cols = df.columns
    # choose columns to include in content: include all but avoid huge binary columns
    # join using ' | ' separator to create readable doc text
    content_expr = concat_ws(" | ", *[col(c).cast("string") for c in cols])
    # build kb_df
    kb_df = df.select(content_expr.alias("content")).limit(max_rows).withColumn("id", concat_ws("_", col("content").substr(1,50), col("content").substr(51,50)))
    # Reorder to (id, content)
    kb_df = kb_df.select("id", "content")
    # write to UC Delta table
    kb_df.write.mode("overwrite").saveAsTable(dest_table)
    print(f"Wrote KB docs to {dest_table} ({kb_df.count()} rows)")
    return dest_table

# ---------------- helper: enable CDF on KB table ----------------
def enable_cdf(table_name: str):
    try:
        spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
        print("Enabled Change Data Feed on", table_name)
    except Exception as e:
        print("Failed to set CDF on", table_name, "— check privileges. Error:", e)

# ---------------- helper: create delta sync index if missing -------------
def create_delta_sync_index(index_name: str = INDEX_NAME, source_table: str = KB_DOCS_TABLE, embedding_model: str = EMBED_MODEL):
    try:
        idx = vclient.get_index(index_name)
        print("Index exists:", index_name)
        return idx
    except Exception:
        print("Index not found; attempting to create:", index_name)
    # Try common SDK signature (may vary by runtime)
    try:
        idx = vclient.create_index(index_name, source=source_table, embedding_model=embedding_model, sync_mode="AUTO")
        print("Created index (AUTO sync):", index_name)
        return idx
    except TypeError:
        try:
            idx = vclient.create_index(index_name, source_table, embedding_model)
            print("Created index (fallback signature):", index_name)
            return idx
        except Exception as e:
            print("Index creation failed; create the index via UI if needed. Error:", e)
            raise

# ---------------- Databricks-native retriever wrapper -------------------
class DatabricksVectorRetriever:
    """
    Wrapper for Databricks Vector Search index querying (new SDK-compatible).
    """
    def __init__(self, index_name: str):
        # ensure non-empty and normalized index name
        index_name = (index_name or "").strip()
        if not index_name:
            raise AssertionError(
                "INDEX_NAME is empty. Set INDEX_NAME to your full index name, "
                "for example: 'hive_metastore.demo.retail_media_kb_index' or 'main.demo.retail_media_kb_index'."
            )
        self.index_name = index_name
        self.client = VectorSearchClient()
        self.index = None


    def _get_index(self):
        if self.index is None:
            self.index = self.client.get_index(self.index_name)
        return self.index

    def query(self, text: str, k: int = 3):
        index = self._get_index()
        try:
            resp = index.query(query_text=text, num_results=k)
        except TypeError:
            resp = index.query(query=text, k=k)

        # New API (runtime 14.x+): use index.query(query_text=..., num_results=...)
        try:
            resp = index.query(query_text=text, num_results=k)
        except TypeError:
            # Some SDKs use slightly different param names
            resp = index.query(query=text, k=k)

        # Normalize hits
        results = []
        hits = []
        if isinstance(resp, dict):
            hits = resp.get("result", {}).get("data_array") or resp.get("results") or resp.get("hits") or []
        else:
            # object-style return
            hits = getattr(resp, "results", getattr(resp, "hits", [])) or []

        for h in hits:
            payload = h.get("payload", {}) if isinstance(h, dict) else getattr(h, "payload", {})
            content = ""
            if isinstance(payload, dict):
                content = (
                    payload.get("content")
                    or payload.get("text")
                    or payload.get("payload_content")
                    or ""
                )
            else:
                content = getattr(h, "payload_content", "")
            results.append(
                {
                    "id": h.get("id") if isinstance(h, dict) else getattr(h, "id", None),
                    "content": content,
                    "score": h.get("score") if isinstance(h, dict) else getattr(h, "score", None),
                    "payload": payload,
                }
            )
        return results


# ---------------- Build supervisor (no Genie) ---------------------------
def stringify_content(state):
    msgs = state["messages"]
    if isinstance(msgs[-1].content, list):
        msgs[-1].content = json.dumps(msgs[-1].content, indent=4)
    return {"messages": msgs}

def create_langgraph_supervisor(llm: Runnable, externally_served_agents: List[ServedSubAgent] = None, in_code_agents: List[InCodeSubAgent] = None) -> CompiledStateGraph:
    if externally_served_agents is None:
        externally_served_agents = []
    if in_code_agents is None:
        in_code_agents = []
    agents = []
    agent_descriptions = ""
    from databricks_langchain import UCFunctionToolkit
    for a in in_code_agents:
        agent_descriptions += f"- {a.name}: {a.description}\n"
        uc_toolkit = UCFunctionToolkit(function_names=a.tools)
        agents.append(create_agent(llm, tools=uc_toolkit.tools, name=a.name))
    for a in externally_served_agents:
        agent_descriptions += f"- {a.name}: {a.description}\n"
        model = ChatDatabricks(endpoint=a.endpoint_name, use_responses_api=("responses" in (a.task or "")))
        model._stream = lambda x: model._stream(**x, stream=False)
        agents.append(create_agent(model, tools=[], name=a.name, post_model_hook=stringify_content))
    prompt = f"""
You are a supervisor in a multi-agent system.

1. Understand the user's last request.
2. Read through the entire chat history.
3. If the answer to the user's last request is present in chat history, answer using information in the history.
4. If the answer is not in the history, from the below list of agents, determine which agent is best suited to answer the question.
5. Provide a summarized response to the user's last query, even if it's been answered before.

{agent_descriptions}
"""
    compiled = create_supervisor(agents=agents, model=llm, prompt=prompt, add_handoff_messages=False, output_mode="full_history").compile()
    return compiled

# ---------------- Run supervisor with KB context -------------------------
def run_supervisor_with_kb_context(compiled_supervisor: CompiledStateGraph, retriever: DatabricksVectorRetriever, user_question: str, kb_top_k: int = KB_TOP_K, print_output: bool = True):
    hits = retriever.query(user_question, k=kb_top_k)
    if hits:
        ctx_pieces = []
        for h in hits:
            c = h.get("content") or ""
            if len(c) > 1200:
                c = c[:1200] + " ...[truncated]"
            doc_id = h.get("id") or h.get("payload", {}).get("id", "unknown")
            score = h.get("score")
            ctx_pieces.append(f"ID:{doc_id} (score:{score}) - {c}")
        kb_context = "\n---\n".join(ctx_pieces)
    else:
        kb_context = ""
    messages = []
    if kb_context:
        messages.append({"role": "system", "content": f"KB_CONTEXT:\n{kb_context}"})
    messages.append({"role": "user", "content": user_question})
    for update_key, events in compiled_supervisor.stream({"messages": messages}, stream_mode=["updates"]):
        for node_name, node_data in events.items():
            msgs = node_data.get("messages", [])
            for m in msgs:
                mid = getattr(m, "id", m.get("id") if isinstance(m, dict) else None)
                role = getattr(m, "role", m.get("role") if isinstance(m, dict) else None)
                content = getattr(m, "content", m.get("content") if isinstance(m, dict) else None)
                if print_output:
                    print(f"[node={node_name}] role={role} id={mid}")
                    if isinstance(content, list):
                        try:
                            print(json.dumps(content, indent=2))
                        except Exception:
                            print(content)
                    else:
                        print(content)
    return True

# ---------------- Example usage (main) ----------------------------------
if __name__ == "__main__":
    # 1) Build KB docs table from demo.retail_media
    print("Building KB docs table from", SOURCE_TABLE, "->", KB_DOCS_TABLE)
    build_kb_docs_from_source(SOURCE_TABLE, KB_DOCS_TABLE, max_rows=10000)
    enable_cdf(KB_DOCS_TABLE)

    # 2) Create or verify Delta Sync index (managed embeddings)
    try:
        create_delta_sync_index(index_name=INDEX_NAME, source_table=KB_DOCS_TABLE, embedding_model=EMBED_MODEL)
    except Exception as e:
        print("Index creation/verification error (proceeding to query; ensure index exists). Error:", e)

    # 3) Compile supervisor with in-code agents you need
    llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
    in_code_agents = [
        InCodeSubAgent(tools=[], name="sql-generator-agent", description="Generates SQL from NL/JSON plans."),
        InCodeSubAgent(tools=[], name="code-execution-agent", description="Executes code snippets and returns results."),
    ]
    served_agents = []  # add any served endpoints if needed
    print("Compiling supervisor...")
    compiled = create_langgraph_supervisor(llm, served_agents, in_code_agents)
    print("Supervisor compiled.")

    # 4) Instantiate retriever and run a test question
    retriever = DatabricksVectorRetriever(index_name=INDEX_NAME)
    user_q = "Total revenue by month for 2024 for completed orders."
    print("\nRunning supervisor with KB context (top-k from Databricks Vector Index)...\n")
    run_supervisor_with_kb_context(compiled, retriever, user_q, kb_top_k=KB_TOP_K)
    print("\nDone.")


In [0]:
# Replace with available tools or remove unavailable ones
in_code_agents = [
    # Remove or replace 'system.ai.sql_generator' with a valid function name
    InCodeSubAgent(
        tools=["system.ai.run_code"], 
        name="code-execution-agent", 
        description="Executes code snippets and returns results."
    ),
]